# MicroDuck keyboard teleop: policy inference check

This lab loads a trained `model_*.pt`, creates **one MicroDuck**, and runs policy inference in MuJoCo at **50 Hz**. The keyboard does not drive joints directly; it changes the desired velocity `(vx, vy, yaw_rate)`. The PPO actor turns live observations into 14-D joint actions.

```text
┌─→ keyboard velocity command
│        │
│        ▼
│   61-D observation → PPO actor → 14-D joint targets
│        │
│        ▼
│   BAM actuator → MuJoCo → new pose, velocity, sensors
│        │
└────────┘
```

This checks that the checkpoint loads, observation dims match, the actor can infer in real time, actions drive the robot, and the policy responds in closed loop to human velocity commands.

> This is still a **simulation inference check**, not hardware deployment. Hardware still needs export/numeric consistency, device I/O, control period, latency, action limits, fall protection, and e-stop validation.

## Prerequisites

1. Start the lab with the updated image (browser MuJoCo desktop included);
2. First use the bundled, auto-selected 950-iter reference demo to verify the teleop stack and set a “normal response” baseline;
3. Then load the 300-iter model trained in `01_velocity_lab.ipynb` and compare stability and tracking under the same commands.


## 1. Verify the teleop stack with the stable reference demo

The next cell loads `examples/velocity_flat_demo/model_950.pt` and creates one MuJoCo env with one robot. This model is not a classroom training artifact. It was directionally fine-tuned, then auto-selected from several checkpoints with a six-direction fixed-command gate:

- 32 parallel trials per direction, 500 steps (10 s);
- Forward, back, left, and right all passed stability, sign, and error gates;
- At `yaw_rate=±0.8 rad/s`, seed-123 left/right means have the requested sign and 100% 10 s survival; cross-seed right turns remain weaker than translation, so this is an improved reference, not a perfect controller.

Confirm:

1. The embedded view and keyboard focus work;
2. Arrow keys and Q/E reach the policy;
3. The robot shows visible responses to forward/back, lateral, and yaw commands;
4. The green target linear-velocity arrow and blue actual linear-velocity arrow appear.

After the reference-demo stack looks healthy, switch to the student model later for contrast. Full numbers live in `examples/velocity_flat_demo/cardinal_eval.json`.


In [ ]:
# Stable baseline: start the auto-selected 950-iter reference demo.
!bash /workspace/microduck_rl_tutorial/scripts/run_teleop.sh demo


In [ ]:
# Confirm the teleop process is actually running, then embed the noVNC live desktop.
# iframe host = Jupyter hostname in the browser bar (including jump hosts); port = noVNC on that host.
import sys

sys.path.insert(0, "/workspace/microduck_rl_tutorial/scripts")
from microduck_novnc_embed import display_teleop_frame

display_teleop_frame(lang="en")


## 2. Keyboard control (absolute-direction mode)

After connecting, double-click the MuJoCo window title bar in the embedded desktop to maximize it, then click the robot view so keyboard focus enters the embed. The four arrow keys set translation; `Q/E` set yaw. WASD no longer drives the robot. If the arrows scroll the notebook, focus is not in the desktop—click the robot view again.

| Key | Full command set `(vx, vy, yaw_rate)` |
|---|---|
| `↑` | `(+0.2, 0, 0)`: forward |
| `↓` | `(-0.2, 0, 0)`: backward |
| `←` | `(0, +0.2, 0)`: left strafe |
| `→` | `(0, -0.2, 0)`: right strafe |
| `Q` | `(0, 0, +0.8)`: turn left |
| `E` | `(0, 0, -0.8)`: turn right |
| `X` or `Enter` | `(0, 0, 0)`: stop |
| `Backspace` | reset the robot and zero the command |
| `Space` | resume/continue; pause-on-space is disabled to avoid accidents |
| `N` | enter pause and step once |
| `+ / -` | playback speed |

Each motion key sets a fixed speed and clears the other axes; repeating a key does not keep accelerating. This mode checks forward/back, lateral, and in-place yaw separately; combined commands are not supported yet.

### How to read the velocity arrows and body frame

- All commands use the **robot body frame**, not screen or world axes: `+vx` is facing direction, `+vy` is the robot’s own left, `+yaw` is counterclockwise from above. After you drag the camera, on-screen left/right may not match body left/right;
- After `↑/↓/←/→`, two arrows appear above the robot: **green** is the keyboard target linear velocity, **blue** is actual body linear velocity from MicroDuck `EntityData`;
- The two arrow origins are offset sideways by about `0.1 m` so the colors do not hide each other. **Do not use the gap between arrows as tracking error**; compare direction and length;
- The blue arrow is instantaneous and will jitter with foot alternation and body sway. Watch whether it generally follows green over a few seconds, not whether every frame overlaps;
- Arrow direction is motion direction; length is speed. Matching direction and similar length means better tracking; a long-term opposite or much shorter blue arrow means poor direction or speed tracking;
- When the command is zeroed the green arrow disappears; the camera keeps following the robot while preserving your chosen view;
- `Q/E` set yaw rate. The current viewer draws linear-velocity arrows only, so in-place turns have no green arrow—watch rotation and `vyaw=±0.80` in the log;
- If arrows are hidden by the robot, drag the view or scroll-zoom in the MuJoCo pane.

> If the log shows `paused` and the robot is frozen, press `Space` once inside the robot view. Browser teleop disables Space as a pause toggle so notebook scrolling does not accidentally freeze the sim.


## 3. Switch to the student’s 300-iter model for contrast

First note how the demo behaves under forward/back, lateral, and yaw commands. Then set `LOAD_STUDENT_MODEL` in the next cell to `True` and run that cell alone. The script finds the latest finished 300-iter run, stops the demo viewer, and loads the matching `model_*.pt`.

Use the same keys and watch response direction, episode duration, drift, jitter, and fall rate. The default is `False`, so **Run All will not switch models by accident**.


In [ ]:
# Keep Run All from accidentally replacing the reference demo.
LOAD_STUDENT_MODEL = False

if LOAD_STUDENT_MODEL:
    !bash /workspace/microduck_rl_tutorial/scripts/run_teleop.sh latest
    print("Student model started. Click Reconnect in the embed, then compare with the same commands.")
else:
    print("Still using the reference demo. To compare, set LOAD_STUDENT_MODEL = True and run this cell alone.")


### How to interpret demo vs student-model differences

This step is not only “which walks better.” It also locates which layer failed:

- **Reference demo responds stably:** checkpoint load, 61-D obs, PPO actor inference, keyboard injection, MuJoCo, and video transport are healthy;
- **Student model barely moves, steps in place, drifts, or falls:** together with that model’s low reward and short episode length in `01`, treat it as an unfinished gait, not a teleop or network outage;
- **Green target arrow jumps with the arrows, but blue arrow and the robot stay weak:** the command arrived; the gap is mostly policy and dynamics;
- **Green arrow does not change and the log has no new command:** re-check embed focus and keyboard delivery.

The reference demo’s improvement is not only “trained longer.” An automatic checkpoint sweep found the old 500-iter model had tiny lateral amplitude and the wrong mean sign on right turns. Later work balanced six-direction sampling, strengthened velocity tracking reward, raised yaw samples and left-right mirror constraints, then selected `model_950` with the fixed-command gate. Rising training reward and the final checkpoint do not replace behavior-oriented acceptance.

This class’s 300-iter checkpoint lasts only about 1–2 s per episode on average, so it is a useful “undertrained counter-example,” not a stable deployable model. Establish a healthy baseline with the reference demo first, then compare the student model, so you do not misread model quality as an inference-stack failure.

> That judgment is for this actual checkpoint. Other hardware, seeds, or training configs can yield a different 300-iter model; re-check metrics and video.


## 4. Stop interactive inference

Closing, scrolling away from, or leaving the notebook only disconnects the view; background inference keeps running. When you finish, set `STOP_TELEOP` in the next cell to `True` and run it alone to free GPU and sim resources. The default is `False`, so **Run All will not black out the embed**.


In [ ]:
# Safety switch: Run All will not close the viewer.
STOP_TELEOP = False

if STOP_TELEOP:
    !bash /workspace/microduck_rl_tutorial/scripts/run_teleop.sh stop
else:
    print("Teleop stays running. When finished, set STOP_TELEOP = True and run this cell alone.")


## 5. Inference checklist

| Check | Pass criterion |
|---|---|
| Checkpoint load | First shows reference demo `model_950.pt`, then student `model_*.pt`, with no dim errors |
| Inference realtime | Viewer stays near realtime, keeps updating after input, no long stalls |
| Target linear velocity | Green arrow changes immediately on arrow keys; blue arrow is actual body velocity; Q/E show no linear-velocity arrow |
| Frame semantics | Forward/back/left/right are body heading, not screen direction |
| Zero command | After zeroing, the robot can stop or stay relatively still instead of drifting forever |
| Forward/back | Motion direction follows the sign of `vx` |
| Lateral | Robot can strafe left/right with the sign of `vy` |
| Yaw | Robot rotates with the sign of `yaw_rate`; weaker yaw tracking than translation is allowed |
| Single-axis absolute commands | Switching motion keys clears the previous axis; no surprise stacking |
| Model gap | Student model may drift, jitter, or fall sooner than the demo, with shorter episodes |
| Disturbance / recovery | No wild oscillation after sudden commands; can restart after reset |

If the checkpoint loads but the robot ignores the keyboard, confirm the embed desktop and MuJoCo window have focus, and watch whether `(vx, vy, vyaw)` changes in `/workspace/runs/logs/teleop.log`.

Next: open [`03_quiz.ipynb`](03_quiz.ipynb) for the five-question lesson check.
